# 02 — Colab: Import, Analyze, Visualize, Build Paper Bundle

Run this after the Kaggle inference notebook has produced `kaggle_results_<profile>.zip`. This notebook performs the statistics and generates the evidence bundle for the future paper.

In [ ]:
REPO_URL = "https://github.com/MichealSK/political-bias-lab.git"
PROFILE = "smoke"
DRIVE_ROOT = "/content/drive/MyDrive/political-bias-lab"
KAGGLE_RESULTS_ZIP = f"{DRIVE_ROOT}/transfer/kaggle_results_{PROFILE}.zip"
REPO_DIR = "/content/political-bias-lab"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

In [ ]:
from src.cloud import link_colab_persistent_dirs
link_colab_persistent_dirs(REPO_DIR, DRIVE_ROOT)

## Import the Kaggle result ZIP

Place the downloaded Kaggle output ZIP at `KAGGLE_RESULTS_ZIP` first. The import helper only extracts `results/` members.

In [ ]:
from pathlib import Path
from src.transfer import import_kaggle_output_bundle
assert Path(KAGGLE_RESULTS_ZIP).exists(), KAGGLE_RESULTS_ZIP
import_kaggle_output_bundle(REPO_DIR, KAGGLE_RESULTS_ZIP)
print("Imported:", KAGGLE_RESULTS_ZIP)

In [ ]:
from src.io_utils import read_json
import subprocess
run_manifest = read_json(Path(REPO_DIR)/"results/manifests/kaggle_inference_manifest.json", {})
current_git = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if run_manifest.get("git_revision"):
    assert run_manifest["git_revision"] == current_git, (
        "Analysis code revision differs from inference revision. Check out:",
        run_manifest["git_revision"],
    )
print("Verified Git revision:", current_git)

In [ ]:
from src.config import load_config
from pathlib import Path
cfg = load_config(Path(REPO_DIR)/"config/default.yaml", Path(REPO_DIR)/f"config/{PROFILE}.yaml")

## Inspect raw error rates before analysis

In [ ]:
import pandas as pd
raw_dir = Path(REPO_DIR)/"results/raw"
for p in sorted(raw_dir.glob("*.parquet")):
    df = pd.read_parquet(p)
    failures = int(df["error"].notna().sum()) if "error" in df.columns else 0
    print(f"{p.name:30s} rows={len(df):6d} failures={failures:4d}")

## Compute statistics and figures

In [ ]:
from src.pipeline import analyze_all
counts = analyze_all(cfg, root=REPO_DIR)
counts

In [ ]:
# Phase 1 summary example
p1 = Path(REPO_DIR)/"results/derived/phase1_summary.csv"
if p1.exists():
    display(pd.read_csv(p1))

In [ ]:
# Phase 2 / 3 / 4 summary tables
for name in ["phase2_summary.csv", "phase3_overall.csv", "phase3_per_class.csv", "phase4_judge_summary.csv"]:
    p = Path(REPO_DIR)/"results/derived"/name
    if p.exists():
        print("\n", name)
        display(pd.read_csv(p))

## Human evaluation for Phase 4

`phase4_human_rating_sheet.csv` is blinded to model/condition labels. Keep `phase4_human_blind_key.parquet` hidden from raters until all ratings are finished. Human ratings are strongly recommended for the final paper.

In [ ]:
sheet = Path(REPO_DIR)/"results/derived/phase4_human_rating_sheet.csv"
key = Path(REPO_DIR)/"results/derived/phase4_human_blind_key.parquet"
print("Rating sheet:", sheet)
print("Secret key:", key)

## Build the immutable paper evidence bundle

In [ ]:
from src.pipeline import build_paper_bundle
bundle_dir = build_paper_bundle(cfg, root=REPO_DIR)
print(bundle_dir)

In [ ]:
import shutil
zip_path = shutil.make_archive(str(Path(DRIVE_ROOT)/f"paper_bundle_{PROFILE}"), "zip", root_dir=bundle_dir)
print("Paper bundle ZIP:", zip_path)

### Before moving from pilot to paper

Freeze the entity/pair files, hypotheses, prompts, profile, Git commit, exclusion rules, and primary metrics. Do not tune the final benchmark after looking at the paper-run outcomes.